# 4068. 会议收益：搜索到二分优化 DP

[原文](https://leetcode.cn/problems/subarrays-distinct-element-sum-of-squares-ii/solution/4068kai-hui-zhuan-qian-mo-yu-ye-zhuan-qi-akyz/)

## DFS

In [ ]:
# 枚举互不重叠的安排；首场之前不计空闲收益。

In [ ]:
# Solution 1. DFS
from typing import List, Optional, Tuple, Dict, Set
from functools import lru_cache

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        orderd_meetings = sorted(meetings, key=lambda x: x[0])
        current_revenue = 0
        current_end = -1
        ans = 0
        @lru_cache(None)
        def dfs(meeting_id:int, current_revenue:int, current_end:int):
            nonlocal ans
            ans = max(ans, current_revenue)
            if meeting_id == len(orderd_meetings):
                return
            for i in range(meeting_id, len(orderd_meetings)):
                if current_end <= orderd_meetings[i][0]:
                    extra_revenue = orderd_meetings[i][0] - current_end if current_end < orderd_meetings[i][0] and current_end >= 0 else 0
                    dfs(i + 1, current_revenue + extra_revenue + orderd_meetings[i][2], orderd_meetings[i][1])
                else:
                    continue

        dfs(0, current_revenue, current_end)
        return ans

## BFS

In [ ]:
# 队列仍枚举安排，方案数可能很大。

In [ ]:
# Solution 2. BFS
from typing import List, Optional, Tuple, Dict, Set
from collections import deque
class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings, key=lambda x: x[0])
        n = len(ordered_meetings)
        queue = deque([])
        visited = set()
        step = 0
        ans = 0
        # (下一场从哪个下标开始找, 当前总收益, 上一场结束时间)
        queue.append((0,0,-1))

        while queue:
            cur = queue.popleft()
            meeting_id, current_revenue, prev_end = cur[0], cur[1], cur[2]
            ans = max(ans, current_revenue)
            for i in range(meeting_id,n):
                if prev_end <= ordered_meetings[i][0]:
                    extra_revenue = (ordered_meetings[i][0] - prev_end) if prev_end>0 else 0
                    next_revenue = extra_revenue + ordered_meetings[i][2] + current_revenue

                    queue.append(
                        (i+1,
                        next_revenue,
                        ordered_meetings[i][1]
                        )
                    )
                else:
                    continue
            step+=1

        return ans

## DP

In [ ]:
# dp[i] 表示最后参加会议 i 的最大收益。

In [ ]:
# Solution 3. DP
from typing import List, Optional, Tuple, Dict, Set

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings, key=lambda x: x[0])
        n = len(ordered_meetings)
        dp = [0] * n
        for i in range(n):
            rev = ordered_meetings[i][2]
            for j in range(i):
                if ordered_meetings[j][1]>ordered_meetings[i][0]:
                    continue
                else:
                    extra_revenue = ordered_meetings[i][0] - ordered_meetings[j][1]
                    cur = dp[j]+extra_revenue + ordered_meetings[i][2]
                rev = max(rev,cur)
            dp[i] = rev
        return max(dp)

## 二分优化 DP

In [ ]:
# 按结束时间排序，二分合法前驱，前缀维护 dp[j]-end[j]。

In [ ]:
# Solution 4. DP OPT1

'''
dp[i] 表示最后选择第 i 场会议时的最大总收益。
接上当前会议后：
空闲收入 = start[i] - end[j]
会议收入 = revenue[i] + dp[j]
总收益 = (dp[j] - end[j]) + (revenue[i] + start[i])
因此，用 prefix_best 保存前缀中最大的 dp[j] - end[j]。
'''

from typing import List, Optional, Tuple, Dict, Set
from bisect import bisect_right

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings,key=lambda x:x[1])
        ends = [x[1] for x in ordered_meetings]
        n = len(ordered_meetings)
        dp = [0] * n
        prefix_best = [0] * n
        for i in range(n):
            # 先考虑只参加当前会议
            rev = ordered_meetings[i][2]
            # 找到最后一个结束时间不晚于当前开始时间的前驱
            j = bisect_right(ends,ordered_meetings[i][0],0, i) - 1
            if j >=0:
                cur = prefix_best[j] + ordered_meetings[i][2] + ordered_meetings[i][0]
                rev = max(cur,rev)
            dp[i] = rev

            # 当前会议自己的前驱分数
            prefix_best[i] = dp[i] - ordered_meetings[i][1]

            # 合并当前值和前一个前缀的最大值
            if i>0:
                prefix_best[i] = max(prefix_best[i],prefix_best[i-1])

        return max(dp)